In [1]:
import os
os.chdir('../')
import warnings
import scanpy as sc
import pandas as pd
import numpy as np

In [3]:
# the data path
main_dir = '/home/wergillius/Project/diffuse_differentiate/'
data_dir = '/home/wergillius/Project/diffuse_differentiate/data/Fetal_reference/'

# Fetal Atlas

In [4]:
# cell info and gene info
cells = pd.read_csv(f"{data_dir}/GSE156793_S1_metadata_cells.txt")
genes = pd.read_csv(f"{data_dir}/GSE156793_S2_Metadata_genes.txt")

/tmp/ipykernel_21997/2436808095.py:2: DtypeWarning: Columns (9,20,22,24) have mixed types. Specify dtype option on import or set low_memory=False.
  cells = pd.read_csv(f"{data_dir}/GSE156793_S1_metadata_cells.txt")


In [5]:
cells

,sample,Exon_reads,Intron_reads,All_reads,RT_group,Organ,Fetus_id,Development_day,Sex,Batch,...,subcluster_umap_2,sub_cluster_id,sub_cluster_name,Matched_MCA_cell_name,MCA_beta,Matched_BCA_cell_name,BCA_beta,BCA_cluster_info,Global_umap_1,Global_umap_2
0,expr2-human-577well.AAGGACGATTTCTTATCGA,70,243,372,Eye_H27552,Eye,H27552,117,M,10,...,-1.743434,1,Eye-Retinal progenitors and Muller glia-1,Muller glia(Retina),0.025435,NaN,NaN,NaN,9.231781,-11.857223
1,expr2-human-577well.AACTAGTTGTGGTCCAGGAG,84,299,458,Eye_H27552,Eye,H27552,117,M,10,...,4.467725,1,Eye-Amacrine cells-1,Amacrine cell_Lamp5_high(Retina),0.101341,NaN,NaN,NaN,-0.009978,9.704188
2,expr2-human-577well.GCAACGTTTCTGATTAAGA,137,700,1005,Eye_H27552,Eye,H27552,117,M,10,...,-0.254757,2,Eye-Amacrine cells-2,Amacrine cell_Tfap2b_high(Retina),0.131173,NaN,NaN,NaN,-0.259648,9.861123
3,expr2-human-577well.AAACCATAGTCCATTATCTA,82,324,469,Eye_H27458,Eye,H27458,129,F,4,...,-1.584072,1,Eye-Retinal progenitors and Muller glia-1,Muller glia(Retina),0.025435,NaN,NaN,NaN,9.002279,-11.873724
4,expr2-human-577well.TCGAGAAGTTAGGCAGATA,76,360,543,Eye_H27552,Eye,H27552,117,M,10,...,-0.074160,2,Eye-Retinal progenitors and Muller glia-2,Muller glia(Retina),0.025437,NaN,NaN,NaN,9.087831,-11.975466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4062975,exp3-human-403well.AGAGTACCTTAGATCTACT,174,176,429,Thymus_H27471,Thymus,H27471,110,F,4,...,1.221047,1,Thymus-Thymocytes-1,T cell_Ms4a4b high(Thymus),0.601736,NaN,NaN,NaN,12.960367,0.713999
4062976,exp3-human-410well.ACAACCTATTCATCTCTGCA,97,199,342,Thymus_H27471,Thymus,H27471,110,F,4,...,3.782746,4,Thymus-Thymocytes-4,Proliferating thymocyte(Thymus),0.720343,NaN,NaN,NaN,13.008845,0.030323
4062977,exp3-human-423well.TTCTCATTGTACTTAACCTT,130,137,311,Thymus_H27471,Thymus,H27471,110,F,4,...,NaN,1,Thymus-Antigen presenting cells-1,Pre T cell(Thymus),1.152779,NaN,NaN,NaN,14.240298,0.762571
4062978,exp3-human-476well.TCCAAGTTATATCCATGACT,104,231,389,Thymus_H27471,Thymus,H27471,110,F,4,...,0.922583,2,Thymus-Thymocytes-2,gdT cell (Thymus),0.040909,NaN,NaN,NaN,12.674088,0.581283


In [7]:
# extracting protein coding genes 
fetal_h5 = os.path.join(data_dir, 'GSE156793_loom_parallel.h5ad')
adata_fetal = sc.read_h5ad(fetal_h5)

In [10]:
existed_obs = adata_fetal.obs.copy()
existed_obs.reset_index(drop=True, inplace=True)

In [15]:
# check the order of cells
merge_df = existed_obs.merge(cells, left_index=True, right_index=True)

In [ ]:
merge_df['All_reads_x'] == merge_df['All_reads_y']

0          True
1          True
2          True
3          True
4          True
           ... 
4062975    True
4062976    True
4062977    True
4062978    True
4062979    True
Length: 4062980, dtype: bool

In [42]:
# the order is correct, now we combine 2 df

In [24]:
overlap_cols = np.intersect1d(cells.columns, existed_obs.columns)
non_overlapping = cells[[col for col in cells.columns if col not in overlap_cols]]
merge_df2 = existed_obs.merge(non_overlapping, left_index=True, right_index=True) 

In [36]:
adata_pcgenes = adata_fetal[:,genes['gene_type'] == "protein_coding"].copy()

In [38]:
adata_pcgenes.var.reset_index(drop=True, inplace=True)

In [39]:
adata_pcgenes.obs = merge_df2

In [43]:
adata_pcgenes.obs

,All_reads,Assay,Batch,Development_day,Exon_reads,Experiment_batch,Fetus_id,Intron_reads,Main_cluster_name,Main_cluster_umap_1,...,subcluster_umap_2,sub_cluster_id,sub_cluster_name,Matched_MCA_cell_name,MCA_beta,Matched_BCA_cell_name,BCA_beta,BCA_cluster_info,Global_umap_1,Global_umap_2
0,372,Nuclei,10,117,70,exp2,H27552,243,Retinal progenitors and Muller glia,1.462221,...,-1.743434,1,Eye-Retinal progenitors and Muller glia-1,Muller glia(Retina),0.025435,NaN,NaN,NaN,9.231781,-11.857223
1,458,Nuclei,10,117,84,exp2,H27552,299,Amacrine cells,0.445410,...,4.467725,1,Eye-Amacrine cells-1,Amacrine cell_Lamp5_high(Retina),0.101341,NaN,NaN,NaN,-0.009978,9.704188
2,1005,Nuclei,10,117,137,exp2,H27552,700,Amacrine cells,0.579049,...,-0.254757,2,Eye-Amacrine cells-2,Amacrine cell_Tfap2b_high(Retina),0.131173,NaN,NaN,NaN,-0.259648,9.861123
3,469,Nuclei,4,129,82,exp2,H27458,324,Retinal progenitors and Muller glia,1.477098,...,-1.584072,1,Eye-Retinal progenitors and Muller glia-1,Muller glia(Retina),0.025435,NaN,NaN,NaN,9.002279,-11.873724
4,543,Nuclei,10,117,76,exp2,H27552,360,Retinal progenitors and Muller glia,1.691502,...,-0.074160,2,Eye-Retinal progenitors and Muller glia-2,Muller glia(Retina),0.025437,NaN,NaN,NaN,9.087831,-11.975466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4062975,429,Nuclei,4,110,174,exp3,H27471,176,Thymocytes,1.884309,...,1.221047,1,Thymus-Thymocytes-1,T cell_Ms4a4b high(Thymus),0.601736,NaN,NaN,NaN,12.960367,0.713999
4062976,342,Nuclei,4,110,97,exp3,H27471,199,Thymocytes,1.933699,...,3.782746,4,Thymus-Thymocytes-4,Proliferating thymocyte(Thymus),0.720343,NaN,NaN,NaN,13.008845,0.030323
4062977,311,Nuclei,4,110,130,exp3,H27471,137,Antigen presenting cells,1.491640,...,NaN,1,Thymus-Antigen presenting cells-1,Pre T cell(Thymus),1.152779,NaN,NaN,NaN,14.240298,0.762571
4062978,389,Nuclei,4,110,104,exp3,H27471,231,Thymocytes,1.471399,...,0.922583,2,Thymus-Thymocytes-2,gdT cell (Thymus),0.040909,NaN,NaN,NaN,12.674088,0.581283


In [45]:
adata_pcgenes.write_h5ad(
    os.path.join(data_dir, 'GSE156793_protein_coding.h5ad'),
)